# Phase 1 · 03 — MCP **from scratch**

So far, tools lived inside the same program that called the model. That doesn't
scale: every app re-declares the same tools.

**MCP (Model Context Protocol)** is a standard so you can publish tools once from
a **server**, and any **client** (a notebook, an IDE, Claude Desktop, an LLM app)
can **discover** and **call** them.

It sounds heavy, but underneath MCP is just:

> **JSON-RPC 2.0 messages sent over a transport.**

Let's prove it by building a server and client by hand — no `mcp` library.

**JSON-RPC 2.0 in one table:**

| Kind | Shape |
|---|---|
| Request | `{"jsonrpc":"2.0", "id":1, "method":"...", "params":{...}}` |
| Response | `{"jsonrpc":"2.0", "id":1, "result":{...}}` (or `"error"`) |
| Notification | like a request but **no `id`** → gets **no response** |

The MCP methods we implement: `initialize` (handshake),
`notifications/initialized`, `tools/list` (discovery), `tools/call` (execution).

## The server logic, as a pure function

`mcp_server_raw.py` contains a `handle_message(msg)` that takes one JSON-RPC
message dict and returns the response dict (or `None` for a notification).
No transport yet — we just call it with plain dicts so the protocol is visible.

In [ ]:
from mcp_server_raw import handle_message, TOOLS
print("server tools:", list(TOOLS))

# 1) Handshake
print("\ninitialize ->")
print(handle_message({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                      "params": {"protocolVersion": "2024-11-05"}}))

# 2) The client's "I'm ready" notification has no id -> returns None (no reply)
print("\nnotifications/initialized ->",
      handle_message({"jsonrpc": "2.0", "method": "notifications/initialized"}))

In [ ]:
import json

# 3) Discovery
resp = handle_message({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
print("tools/list ->")
print(json.dumps(resp["result"], indent=2))

In [ ]:
# 4) Execution
resp = handle_message({
    "jsonrpc": "2.0", "id": 3, "method": "tools/call",
    "params": {"name": "multiply", "arguments": {"a": 6, "b": 7}},
})
print("tools/call multiply(6, 7) ->")
print(json.dumps(resp["result"], indent=2))

## Now over a real transport: stdio

That was pure logic. A real MCP server runs as a **separate process** and speaks
over a **transport** — most commonly **stdio**: newline-delimited JSON on
stdin/stdout. `mcp_server_raw.py` does exactly that in its `serve_stdio()` loop.

Below we launch it as a subprocess and do the full handshake + discovery + call
over the pipe. This is precisely how a tool like Claude Desktop loads a local
MCP server.

In [ ]:
import sys, subprocess, json

proc = subprocess.Popen(
    [sys.executable, "mcp_server_raw.py"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1,
)

def rpc(method, params=None, msg_id=None, notify=False):
    """Send one JSON-RPC message over stdin; read one response line (unless notify)."""
    msg = {"jsonrpc": "2.0", "method": method}
    if not notify:
        msg["id"] = msg_id
    if params is not None:
        msg["params"] = params
    proc.stdin.write(json.dumps(msg) + "\n")
    proc.stdin.flush()
    if notify:
        return None
    return json.loads(proc.stdout.readline())

print("initialize ->", rpc("initialize", {"protocolVersion": "2024-11-05"}, msg_id=1))
rpc("notifications/initialized", notify=True)                      # no reply expected
print("tools/list ->", [t["name"] for t in rpc("tools/list", msg_id=2)["result"]["tools"]])
print("tools/call ->", rpc("tools/call",
      {"name": "add", "arguments": {"a": 40, "b": 2}}, msg_id=3)["result"]["content"])

proc.stdin.close()
proc.terminate()

## Recap

An MCP server is a program that answers JSON-RPC methods
(`initialize`, `tools/list`, `tools/call`) over a transport (stdio). You built
both the message handler and a real stdio round-trip by hand.

The `mcp` **library** does all of this for you — the handshake, schema generation
from type hints, validation, multiple transports — so you write only the tools.

**Next → `04_mcp_with_library.ipynb`:** the same server, in a fraction of the
code, plus wiring the MCP tools to the LLM.